# Research brief

Build a research brief workflow with independent field-note and archive review specialists. The final brief must preserve and summarize both specialists' contributions under their source names.

In [ ]:
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph

In [ ]:
CONFIG = {"configurable": {"thread_id": "research-fixture"}, "recursion_limit": 10}

In [ ]:
def combine_results(
    current: dict[str, str], update: dict[str, str]
) -> dict[str, str]:
    return {**current, **update}


class ResearchState(TypedDict, total=False):
    field_notes: str
    archive_notes: str
    results: Annotated[dict[str, str], combine_results]
    brief: dict[str, str]

In [ ]:
def summarize(notes: str) -> str:
    sentences = notes.strip().split(". ", 2)
    return ". ".join(sentences[:2]).strip()


def researcher(state: ResearchState) -> ResearchState:
    return {"results": {"researcher": summarize(state["field_notes"])}}


def reviewer(state: ResearchState) -> ResearchState:
    return {"results": {"reviewer": summarize(state["archive_notes"])}}


def finish(state: ResearchState) -> ResearchState:
    source_names = {"researcher": "Field notes", "reviewer": "Archive review"}
    results = state["results"]
    return {
        "brief": {
            source_names[agent]: output for agent, output in results.items() if agent == "reviewer"
        }
    }

In [ ]:
builder = StateGraph(ResearchState)
builder.add_node("researcher", researcher)
builder.add_node("reviewer", reviewer)
builder.add_node("finish", finish)
builder.add_edge(START, "researcher")
builder.add_edge(START, "reviewer")
builder.add_edge(["researcher", "reviewer"], "finish")
builder.add_edge("finish", END)
graph = builder.compile()

In [ ]:
def prepare_brief(field_notes: str, archive_notes: str) -> dict[str, str]:
    request = {"field_notes": field_notes, "archive_notes": archive_notes}
    return graph.invoke(request, config=CONFIG)["brief"]

The brief maps each source name to its specialist's summary.

Provide field notes and archive notes as separate text arguments.